# Method 3: Incorporation of Adjacency Constraint with Manual Assignment of Northern Districts

## Method Details:

#### Objective Function:

- Minimize the lack of compactness (a.k.a, the sum of distances between counties that are assigned to the same district).

#### Constraints:

1. **One-County-One-District:** Each county should be assigned to exactly one district.
2. **Ideal Population:** The population in each district should be approximately equal (+- 20%) to the ideal district population size.
3. **Adjacency:** Each county in a district should be adjacency to at least one of the other counties also in that district.

#### The 3 Large Population Counties Consideration
Given that there are three counties that contain very high population around Detroit, these districts will manuaglly be assigned to their own districts.

***Approach to this consideration***
1. **Adjust the Dataset**
- Remove the three large counties, Macomb, Oakland, and Wayne from the dataset, since they will manually be assigned to their own districts.
2. **Adjust the number of districts and recalculate the average distict population**
- Since we are removing the three largest counties, we will be to recalculate the average size we want our disticts to be based on the remaining counties.
3. **Modify and Re-run the Model**
- An additional modification I made when re-running this model, what I adjusted the district population constraint to be +/- 20%. This was the only way I was able to find an optimal solution.

#### Manual Assignment of Northern Most Counties into Districts

**Manual Assignment of District 11: (Most Northern Counties)**
1. Alger
2. Baraga
3. Chippewa
4. Delta
5. Dickinson
6. Gogebic
7. Houghton
8. Iton
9. Keweenaw
10. Luce
11. Mackinac
12. Marquette
13. Menominee
14. Ontonagan
15. Schoolcraft
- Total population: 301397

**Manual Assignment of District 10: (2nd Most Northwen Counties)**
1. Emmet
2. Cheboygan
3. Presque Isle
4. Charlevoix
5. Antrim
6. Otsego
7. Montomorency
8. Alpena
- Total population: 188066


### Step 1: Load necessary libraries

In [17]:
# Import necessary libraries
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt   
import pandas as pd 
from math import pi, pow, sin, cos, asin, sqrt, floor
from pulp import *

### Step 2: Read in the Excel file and GeoJSON file
- Then merge the two data sources to have a comprehensive dataset.
    - In the 'michigan_counties' dataframe, the county name column is 'county_names', and in the 'michigan_counties_geojson' datafram, the county name column is 'name'.

In [18]:
# Read in the Excel file of Michigan counties
xlsx_file_path = '/Users/stefanjenss/Desktop/DataScience/Decision_analytics/Module6/michigan_counties_mod.xlsx'
michigan_counties = pd.read_excel(xlsx_file_path)

# Read in the GeoJSON file of Michigan county boundaries
geojson_file_path = '/Users/stefanjenss/Desktop/DataScience/Decision_analytics/Module6/michigan_counties_mod.geojson'
michigan_counties_geojson = gpd.read_file(geojson_file_path)

print(michigan_counties.head())
print("")
print(michigan_counties_geojson.head())

   count_id county_names   latitude  longitude  pop2020
0         0     Leelanau  45.151771 -86.038496    22870
1         1      Clinton  42.943652 -84.601517    79748
2         2      Wexford  44.338367 -85.578414    34196
3         3       Branch  41.916119 -85.059044    44531
4         4        Ionia  42.945094 -85.074603    66809

                                    geo_point_2d statefp countyfp  countyns  \
0   {'lon': -86.0384960523, 'lat': 45.151770859}      26      089  01622987   
1  {'lon': -84.6015165533, 'lat': 42.9436523662}      26      037  01622961   
2  {'lon': -85.5784138137, 'lat': 44.3383668115}      26      165  01623023   
3  {'lon': -85.0590443604, 'lat': 41.9161186535}      26      023  01622954   
4  {'lon': -85.0746031181, 'lat': 42.9450938315}      26      067  01622976   

   geoid      name         namelsad stusab lsad classfp  ... cbsafp metdivfp  \
0  26089  Leelanau  Leelanau County     MI   06      H1  ...  45900     None   
1  26037   Clinton   Clinton

In [19]:
# Merge the two dataframes on the county name
michigan_counties_merged = michigan_counties.merge(michigan_counties_geojson, left_on='county_names', right_on='name', how='inner')
michigan_counties_merged.head()

,count_id,county_names,latitude,longitude,pop2020,geo_point_2d,statefp,countyfp,countyns,geoid,...,cbsafp,metdivfp,funcstat,aland,awater,intptlat,intptlon,state_name,countyfp_nozero,geometry
0,0,Leelanau,45.151771,-86.038496,22870,"{'lon': -86.0384960523, 'lat': 45.151770859}",26,089,01622987,26089,...,45900,None,A,899241895,5659105307,+45.1461816,-086.0515740,Michigan,89,"POLYGON ((-85.56175 44.95226, -85.56209 44.950..."
1,1,Clinton,42.943652,-84.601517,79748,"{'lon': -84.6015165533, 'lat': 42.9436523662}",26,037,01622961,26037,...,29620,None,A,1467017475,21098128,+42.9504550,-084.5916949,Michigan,37,"POLYGON ((-84.83762 43.03264, -84.83754 43.032..."
2,2,Wexford,44.338367,-85.578414,34196,"{'lon': -85.5784138137, 'lat': 44.3383668115}",26,165,01623023,26165,...,15620,None,A,1463148726,27182043,+44.3313751,-085.5700462,Michigan,165,"POLYGON ((-85.81909 44.42450, -85.81910 44.425..."
3,3,Branch,41.916119,-85.059044,44531,"{'lon': -85.0590443604, 'lat': 41.9161186535}",26,023,01622954,26023,...,17740,None,A,1311605515,34420092,+41.9184551,-085.0668852,Michigan,23,"POLYGON ((-85.29293 41.98482, -85.29293 41.984..."
4,4,Ionia,42.945094,-85.074603,66809,"{'lon': -85.0746031181, 'lat': 42.9450938315}",26,067,01622976,26067,...,24340,None,A,1479710906,22590318,+42.9446503,-085.0737660,Michigan,67,"POLYGON ((-85.07503 43.12021, -85.06470 43.120..."


### Step 3: Calculate the Distance Between County Pairs

In [20]:
# Define function to calculate distance between two sets of longitudes / latitudes

# Function to convert degrees to radians
def degrees_to_radians(x):
    return((pi / 180) * x)

# Function to calculate distance between two points on a sphere (in miles) given their longitudes and latitudes
def lon_lat_distance_miles(lon_a, lat_a, lon_b, lat_b):
    """
    Calculates the great-circle distance between two points on a sphere given their longitudes and latitudes.
    
    Parameters:
    lon_a (float): longitude of point A in degrees
    lat_a (float): latitude of point A in degrees
    lon_b (float): longitude of point B in degrees
    lat_b (float): latitude of point B in degrees
    
    Returns:
    float: distance between the two points in miles
    """
    radius_of_earth = 24872 / (2 * pi)
    c = sin((degrees_to_radians(lat_a) - \
    degrees_to_radians(lat_b)) / 2)**2 + \
    cos(degrees_to_radians(lat_a)) * \
    cos(degrees_to_radians(lat_b)) * \
    sin((degrees_to_radians(lon_a) - \
    degrees_to_radians(lon_b))/2)**2
    return(2 * radius_of_earth * (asin(sqrt(c))))

# Function to convert the distance between two points on a sphere (in miles) to meters
def lon_lat_distance_meters (lon_a, lat_a, lon_b, lat_b):
    return(lon_lat_distance_miles(lon_a, lat_a, lon_b, lat_b) * 1609.34)

In [21]:
# Remove population to allow easy joining of long and lat for each county pair
lat_lon = ['county_names', 'latitude', 'longitude']
lat_lon = michigan_counties_merged[lat_lon]

# Create list of county names for pairing        
county_names = michigan_counties['county_names'].to_numpy()

# Create each unique pair
pairs = []

# Loop through each county name and create a pair with each other county name
for i in range(len(county_names)):
    for j in range(i + 1, len(county_names)):
        pairs.append((county_names[i], county_names[j]))

# Create column names for county pairs df
col_names = ['county_1', 'county_2']

# Create df of county pairs                
county_pairs = pd.DataFrame(pairs, columns = col_names)

 # Add first county longitude and latitude
county_pairs = county_pairs.merge(lat_lon, left_on = 'county_1', right_on = 'county_names', how = 'left')
county_pairs.drop('county_names', axis = 1, inplace = True) # Drop county names column
county_pairs = county_pairs.rename(columns={'latitude': 'county_1_lat', 'longitude': 'county_1_long'}) # Rename columns

# Add second county longitude and latitude
county_pairs = county_pairs.merge(lat_lon, left_on = 'county_2', right_on = 'county_names', how = 'left')
county_pairs.drop('county_names', axis = 1, inplace = True) # Drop county names column
county_pairs = county_pairs.rename(columns={'latitude': 'county_2_lat', 'longitude': 'county_2_long'}) # Rename columns

# Add distance between each county pair in miles and meters;
distance_miles = [] # Create empty list to store distance in miles
distance_meters = [] # Create empty list to store distance in meters

# Loop through each county pair and calculate distance in miles and meters
for i in range(len(county_pairs)):
    distance_miles.append(lon_lat_distance_miles(county_pairs.iloc[i, 2], county_pairs.iloc[i, 3], county_pairs.iloc[i, 4], county_pairs.iloc[i, 5]))
    distance_meters.append(lon_lat_distance_meters(county_pairs.iloc[i, 2], county_pairs.iloc[i, 3], county_pairs.iloc[i, 4], county_pairs.iloc[i, 5]))

# Add distance columns to county pairs df
county_pairs['distance_miles'] = distance_miles
county_pairs['distance_meters'] = distance_meters

# Check table
county_pairs

,county_1,county_2,county_1_lat,county_1_long,county_2_lat,county_2_long,distance_miles,distance_meters
0,Leelanau,Clinton,45.151771,-86.038496,42.943652,-84.601517,100.038253,160995.562830
1,Leelanau,Wexford,45.151771,-86.038496,44.338367,-85.578414,32.050066,51579.452482
2,Leelanau,Branch,45.151771,-86.038496,41.916119,-85.059044,69.831366,112382.410744
3,Leelanau,Ionia,45.151771,-86.038496,42.945094,-85.074603,67.621439,108825.886417
4,Leelanau,Mecosta,45.151771,-86.038496,43.640768,-85.324634,49.938224,80367.581755
...,...,...,...,...,...,...,...,...
3398,Arenac,Alcona,44.042885,-83.747242,44.683623,-83.129008,43.010988,69219.303841
3399,Arenac,Livingston,44.042885,-83.747242,42.602917,-83.911528,15.593543,25095.312007
3400,Charlevoix,Alcona,45.502498,-85.373250,44.683623,-83.129008,155.151831,249692.047763
3401,Charlevoix,Livingston,45.502498,-85.373250,42.602917,-83.911528,102.674475,165238.138922


### Step 4: Create an adjacency dictionary for Michigan counties from an existing adjacency .txt file from the census website

4.1 Read in the .txt file with the adjacency information

In [22]:
# Read in the .txt file of county adjacencies
txt_file_path = '/Users/stefanjenss/Desktop/DataScience/Decision_analytics/Module6/county_adjacency_mod.txt'

# Read all lines of the .txt file into a list
with open(txt_file_path, encoding='latin-1') as f:
    county_adjacency = f.readlines()

# Remove \t and \n from each line
county_adjacency = [line.replace('\t', ' ') for line in county_adjacency]
county_adjacency = [line.replace('\n', '') for line in county_adjacency]

county_adjacency

['"Autauga County, AL" 01001 "Autauga County, AL" 01001',
 '  "Chilton County, AL" 01021',
 '  "Dallas County, AL" 01047',
 '  "Elmore County, AL" 01051',
 '  "Lowndes County, AL" 01085',
 '  "Montgomery County, AL" 01101',
 '"Baldwin County, AL" 01003 "Baldwin County, AL" 01003',
 '  "Clarke County, AL" 01025',
 '  "Escambia County, AL" 01053',
 '  "Mobile County, AL" 01097',
 '  "Monroe County, AL" 01099',
 '  "Washington County, AL" 01129',
 '  "Escambia County, FL" 12033',
 '"Barbour County, AL" 01005 "Barbour County, AL" 01005',
 '  "Bullock County, AL" 01011',
 '  "Dale County, AL" 01045',
 '  "Henry County, AL" 01067',
 '  "Pike County, AL" 01109',
 '  "Russell County, AL" 01113',
 '  "Clay County, GA" 13061',
 '  "Quitman County, GA" 13239',
 '  "Stewart County, GA" 13259',
 '"Bibb County, AL" 01007 "Bibb County, AL" 01007',
 '  "Chilton County, AL" 01021',
 '  "Hale County, AL" 01065',
 '  "Jefferson County, AL" 01073',
 '  "Perry County, AL" 01105',
 '  "Shelby County, AL" 01

4.2 Store the Michican counties and their adjaciencies into a list

In [23]:
michigan_adjacency = [] # Create empty list to store county adjacency data
with open(txt_file_path, "r", encoding="ISO-8859-1") as file:
    current_county = None
    for line in file:
        parts = line.strip().split('\t')
        # Check if the county is from Michigan
        if "MI" in parts[0]:
            # If it's a new county
            if len(parts) == 4 and parts[0] == parts[2]:
                current_county = parts[0]
            if current_county:
                michigan_adjacency.append([current_county, parts[len(parts) - 2]])

michigan_adjacency[:10]

[['"Alcona County, MI"', '"Alcona County, MI"'],
 ['"Alcona County, MI"', '"Alpena County, MI"'],
 ['"Alcona County, MI"', '"Iosco County, MI"'],
 ['"Alcona County, MI"', '"Montmorency County, MI"'],
 ['"Alcona County, MI"', '"Ogemaw County, MI"'],
 ['"Alcona County, MI"', '"Oscoda County, MI"'],
 ['"Alger County, MI"', '"Alger County, MI"'],
 ['"Alger County, MI"', '"Delta County, MI"'],
 ['"Alger County, MI"', '"Keweenaw County, MI"'],
 ['"Alger County, MI"', '"Luce County, MI"']]

4.3 Convert these adjacencies in the list into a dictionary format with each county as a key and its adjacent counties as values

In [24]:
# Converting the adjacencies into a dictionary format
adjacency_dict = {} # Create empty dictionary to store adjacency data
for county, adjacent in michigan_adjacency:
    if county not in adjacency_dict:
        adjacency_dict[county] = []
    adjacency_dict[county].append(adjacent)

# Check dictionary
dict(list(adjacency_dict.items())[0:3])

{'"Alcona County, MI"': ['"Alcona County, MI"',
  '"Alpena County, MI"',
  '"Iosco County, MI"',
  '"Montmorency County, MI"',
  '"Ogemaw County, MI"',
  '"Oscoda County, MI"'],
 '"Alger County, MI"': ['"Alger County, MI"',
  '"Delta County, MI"',
  '"Keweenaw County, MI"',
  '"Luce County, MI"',
  '"Marquette County, MI"',
  '"Schoolcraft County, MI"',
  '"Lake County, IL"',
  '"Allegan County, MI"',
  '"Barry County, MI"',
  '"Kalamazoo County, MI"',
  '"Kent County, MI"',
  '"Ottawa County, MI"',
  '"VanBuren County, MI"',
  '"Alcona County, MI"',
  '"Alpena County, MI"',
  '"Montmorency County, MI"',
  '"Oscoda County, MI"',
  '"PresqueIsle County, MI"'],
 '"Antrim County, MI"': ['"Antrim County, MI"',
  '"Charlevoix County, MI"',
  '"Crawford County, MI"',
  '"GrandTraverse County, MI"',
  '"Kalkaska County, MI"',
  '"Leelanau County, MI"',
  '"Otsego County, MI"']}

4.4 Modify the adjacency dictionary so that each county is not being marked as being adjacent to itself

In [25]:
# Remove each county from its own adjacency list
for county, adjacent in adjacency_dict.items():
    adjacency_dict[county] = [x for x in adjacent if x != county]

# Check the first few counties
dict(list(adjacency_dict.items())[0:3])

{'"Alcona County, MI"': ['"Alpena County, MI"',
  '"Iosco County, MI"',
  '"Montmorency County, MI"',
  '"Ogemaw County, MI"',
  '"Oscoda County, MI"'],
 '"Alger County, MI"': ['"Delta County, MI"',
  '"Keweenaw County, MI"',
  '"Luce County, MI"',
  '"Marquette County, MI"',
  '"Schoolcraft County, MI"',
  '"Lake County, IL"',
  '"Allegan County, MI"',
  '"Barry County, MI"',
  '"Kalamazoo County, MI"',
  '"Kent County, MI"',
  '"Ottawa County, MI"',
  '"VanBuren County, MI"',
  '"Alcona County, MI"',
  '"Alpena County, MI"',
  '"Montmorency County, MI"',
  '"Oscoda County, MI"',
  '"PresqueIsle County, MI"'],
 '"Antrim County, MI"': ['"Charlevoix County, MI"',
  '"Crawford County, MI"',
  '"GrandTraverse County, MI"',
  '"Kalkaska County, MI"',
  '"Leelanau County, MI"',
  '"Otsego County, MI"']}

4.4 Modify this list so that only the county name e.g., "Alcona", and not the full name, e.g., "Alcona County, MI"

In [26]:
# Modify the adjacency dictionary to include only the county name
# Modifying the adjacency dictionary to retain only the county name
modified_adjacency_dict = {}

for full_name, adjacents in adjacency_dict.items():
    county_name = full_name.split()[0].replace('"', '')
    modified_adjacency_dict[county_name] = [x.split()[0].replace('"', '') for x in adjacents]

# Checking the first few entries in the modified adjacency dictionary
dict(list(modified_adjacency_dict.items())[:3])


{'Alcona': ['Alpena', 'Iosco', 'Montmorency', 'Ogemaw', 'Oscoda'],
 'Alger': ['Delta',
  'Keweenaw',
  'Luce',
  'Marquette',
  'Schoolcraft',
  'Lake',
  'Allegan',
  'Barry',
  'Kalamazoo',
  'Kent',
  'Ottawa',
  'VanBuren',
  'Alcona',
  'Alpena',
  'Montmorency',
  'Oscoda',
  'PresqueIsle'],
 'Antrim': ['Charlevoix',
  'Crawford',
  'GrandTraverse',
  'Kalkaska',
  'Leelanau',
  'Otsego']}

### Step 5: Modify the DataFrame to exclude the counties that will be manually assigned

5.1 Define the parameters for the problem

In [32]:
# Create a dictionary of county names and their populations
county_populations = michigan_counties_merged[['name', 'pop2020']].set_index('name').to_dict()['pop2020']

# Number of counties and districts in Michigan
n_counties = 83
n_districts = 14

5.2 Create lists of the large counties and the northern counites that will be manually assigned to a district

In [33]:
# Create a list of the three large counties
large_counties = ['Wayne', 'Oakland', 'Macomb']

# Create a list of the most northern counties
most_northern = ["Alger", "Baraga", "Chippewa", "Delta", "Dickinson", "Gogebic", "Houghton", "Iron", "Keweenaw", 
                 "Luce", "Mackinac", "Marquette", "Menominee", "Ontonagon", "Schoolcraft"]

# Create a list of the second most northern counties
second_most_northern = ["Emmet", "Cheboygan", "Presque Isle", "Charlevoix", "Antrim", "Otsego", 
                        "Montmorency", "Alpena"]

# Create a list of all the manually assigned counties
manual_counties = large_counties + most_northern + second_most_northern

# Create a new dataframe that doesn't contain any of the manual counties
michigan_counties_adjusted = michigan_counties_merged[~michigan_counties_merged['name'].isin(manual_counties)]


5.3 Adjust the number of counties and districts, and calculate the new ideal district population

In [34]:
# Adjust the number of counties and districts in Michigan
n_counties_adjusted = len(michigan_counties_adjusted)
n_districts_adjusted = n_districts - (len(large_counties) + 2)

# Approximate population of each district
average_district_population_adjusted = michigan_counties_adjusted['pop2020'].sum() / n_districts_adjusted

5.4 Filter the pairs to exclude the manual counties

In [35]:
# Filter the county pairs dataframe to only include counties that are not in the large counties list
county_pairs_adjusted = county_pairs[~county_pairs['county_1'].isin(manual_counties)]

# Create a dictionary of the adjusted county names and their populations
county_populations_adjusted = michigan_counties_adjusted[['name', 'pop2020']].set_index('name').to_dict()['pop2020']
county_names_adjusted = michigan_counties_adjusted['name'].to_numpy()

5.5 Create the optimization problem

In [36]:
# Initialize the adjusted model
model_adjusted = LpProblem("Michigan_Redistricting_Adjusted", LpMinimize)

# Decision variable: binary variable indicating whether a county is in a district
x_adjusted = LpVariable.dicts("x_adjusted", ((i, j) for i in county_names_adjusted for j in range(n_districts_adjusted)), cat='Binary')

# Objective Function: Minimize the total distance between counties in the same district
model_adjusted += lpSum(county_pairs_adjusted.iloc[i, 7] * x_adjusted[(county_pairs_adjusted.iloc[i, 0], j)] for i in range(len(county_pairs_adjusted)) for j in range(n_districts_adjusted))

# Constraints: Each county must be assigned to exactly one district
for i in range(len(county_names_adjusted)):
    model_adjusted += lpSum(x_adjusted[(county_names_adjusted[i], j)] for j in range(n_districts_adjusted)) == 1

# Constraints: Each district must have a population within 20% of the average population
for j in range(n_districts_adjusted):
    model_adjusted += lpSum(county_populations_adjusted[county_names_adjusted[i]] * x_adjusted[(county_names_adjusted[i], j)] for i in range(len(county_names_adjusted))) >= 0.80 * average_district_population_adjusted # Lower bound
    model_adjusted += lpSum(county_populations_adjusted[county_names_adjusted[i]] * x_adjusted[(county_names_adjusted[i], j)] for i in range(len(county_names_adjusted))) <= 1.20 * average_district_population_adjusted # Upper bound

KeyError: ('VanBuren', 0)

#### 5.5.1 Addition of the adjacency constraint

**I will use the flow-based approach**

5.5.1.1 Validate that every county in 'county_names_adjusted' exists as a key in 'modified_adjacency_dict'

5.5.1.2 Read in the adjacency file

In [ ]:
# Read all lines of the .txt file into a list
with open(txt_file_path, encoding='latin-1') as f:
    county_adjacency = f.readlines()

# Extract Michigan adjacencies
michigan_adjacency = [line for line in county_adjacency if "MI" in line]

michigan_adjacency

In [ ]:
problematic_counties = ['St. Clair', 'Presque Isle', 'Van Buren', 'St. Joseph', 'Grand Traverse']

for line in michigan_adjacency:
    for county in problematic_counties:
        if county in line:
            print(line)

5.2.1.3 Parse the Michigan Adjacency Date

In [16]:
adjacency_pairs = []
current_county = None

for line in michigan_adjacency:
    parts = line.strip().split('\t')
    
    # Check if the county is in Michigan
    if "MI" in parts[0]:
        # If it's a new county
        if current_county != parts[0]:
            current_county = parts[0]
        
        # Add adjacency pair to the list
        adjacency_pairs.append([current_county, parts[-2]])

adjacency_pairs

AttributeError: 'list' object has no attribute 'strip'

5.2.1.4 Convert Adacency Pairs to Dictionary

In [ ]:
adjacency_dict = {}
for county, adjacent in adjacency_pairs:
    if county not in adjacency_dict:
        adjacency_dict[county] = []
    adjacency_dict[county].append(adjacent)

5.2.1.5 Clean Up the Dictionary

In [ ]:
# Modify the adjacency dictionary to retain only the county name
modified_adjacency_dict = {}
for full_name, adjacents in adjacency_dict.items():
    county_name = full_name.split()[0].replace('"', '')
    modified_adjacency_dict[county_name] = [x.split()[0].replace('"', '') for x in adjacents if x != full_name]

5.2.1.6 Verify that all counties in the problem set ('county_names_adjusted') are present in the adjacecncy dictionary

In [ ]:
missing_counties = [county for county in county_names if county not in modified_adjacency_dict]
print(f"Missing counties: {missing_counties}")

In [ ]:
# Extract all unique Michigan county names from the adjacency file
all_michigan_counties_in_adjacency = set([pair[0] for pair in adjacency_pairs])

# Simplify the names
all_michigan_counties_in_adjacency = {name.split()[0].replace('"', '') for name in all_michigan_counties_in_adjacency}

# Check which counties from your dataset are not in the adjacency list
missing_from_adjacency = [county for county in county_names_adjusted if county not in all_michigan_counties_in_adjacency]

print(f"Counties missing from the adjacency file: {missing_from_adjacency}")

In [ ]:
# Decision variable: binary variable indicating flow between two adjacent counties
f = LpVariable.dicts("flow", 
                     ((i, k, j) for i in county_names_adjusted for k in modified_adjacency_dict[i] for j in range(n_districts_adjusted)), 
                     cat='Binary')


In [ ]:
### Step 1: Define the flow decision variables
# Decision variable: binary variable indicating flow between two adjacent counties
f = LpVariable.dicts("flow",
                    ((i,k,j) for i in county_names_adjusted for k in modified_adjacency_dict[i] for j in range(n_districts_adjusted)),
                    cat='Binary')

5.6 Solve the model and print the results

In [ ]:
# Solve the adjusted model
model_adjusted.solve()

# Print the status of the solution
print("Status:", LpStatus[model_adjusted.status])

# Print the objective function value
print("Objective Function Value:", value(model_adjusted.objective))
print("")

# Print the results by county name and district number, along with the total population in each district and the population of each county
for j in range(n_districts_adjusted):
    print("District", j + 1, ":")
    district_counties = [i for i in range(n_counties_adjusted) if x_adjusted[(county_names_adjusted[i], j)].varValue == 1]
    district_population = sum([county_populations_adjusted[county_names_adjusted[i]] for i in district_counties])
    print("Total Population:", district_population)
    for i in range(len(county_names_adjusted)):
        if x_adjusted[(county_names_adjusted[i], j)].varValue == 1:
            print(county_names_adjusted[i], "Population:", county_populations_adjusted[county_names_adjusted[i]])
    print("")

# Manually assign the most northern counties into district 10 (counties listed in 'most_northern' list)
# Show the total population of the district and the population of each county
print("District 10:")
district_population = sum([county_populations[county] for county in most_northern])
print("Total Population:", district_population)
for county in most_northern:
    print(county, "Population:", county_populations[county])
print("")

# Manually assign the second most northern counties into district 11 (counties listed in 'second_most_northern' list)
# Show the total population of the district and the population of each county
print("District 11:")
district_population = sum([county_populations[county] for county in second_most_northern])
print("Total Population:", district_population)
for county in second_most_northern:
    print(county, "Population:", county_populations[county])
print("")

# Manually assign the three large counties to their own districts
for county in large_counties:
    print("District", j + 4, ":")
    print(county, "Population:", county_populations[county])
    j += 1
    print("")

### Step 6: Visualize the generated districts from the optimization results to get a better understanding of the country partitioning amount districts

In [ ]:
michigan_counties_method3 = michigan_counties_merged.copy()

# Add a district assignment column to the dataframe
michigan_counties_method3['district'] = np.nan

# Assign the district number based on the optimization results
for j in range(n_districts_adjusted):
    district_counties = [i for i in range(n_counties_adjusted) if x_adjusted[(county_names_adjusted[i], j)].varValue == 1]
    for i in district_counties:
        michigan_counties_method3.loc[michigan_counties_method3['name'] == county_names_adjusted[i], 'district'] = j + 1

# Manually assign the most northern counties to district 10
for county in most_northern:
    michigan_counties_method3.loc[michigan_counties_method3['name'] == county, 'district'] = 10

# Manually assign the second most northern counties to district 11
for county in second_most_northern:
    michigan_counties_method3.loc[michigan_counties_method3['name'] == county, 'district'] = 11

# Assign the large counties to their own districts starting at district 12
j = 11
for county in large_counties:
    j += 1
    michigan_counties_method3.loc[michigan_counties_method3['name'] == county, 'district'] = j 

# Convert the district column to an integer
michigan_counties_method3['district'] = michigan_counties_method3['district'].astype(int)

# Show the results
michigan_counties_method3[['name', 'district']].sort_values(by = 'district')


5.2 Plot the results

In [ ]:
# Convert the 'geometry' column to a GeoSeries and create a GeoDataFrame
geo_michigan_counties_method3 = gpd.GeoDataFrame(michigan_counties_method3, geometry=gpd.GeoSeries(michigan_counties_method3['geometry']))

# Plot the results
fig, ax = plt.subplots(1, 1, figsize=(10, 10))
geo_michigan_counties_method3.plot(column='district', ax=ax, legend=True, cmap='tab20', legend_kwds={'label': "District Number"})
ax.set_title('Michigan Congressional Districts (Method 3)')

# Annotate each county with its assigned district number
for index, row in geo_michigan_counties_method3.iterrows():
    plt.annotate(text=row['district'], xy=(row['geometry'].centroid.x, row['geometry'].centroid.y), horizontalalignment='center', fontsize=6)

# Annotate each country with its name
for index, row in geo_michigan_counties_method3.iterrows():
    plt.annotate(text=row['name'], xy=(row['geometry'].centroid.x, row['geometry'].centroid.y+.12), horizontalalignment='center', fontsize=5)

# Annotate each county with its population
for index, row in geo_michigan_counties_method3.iterrows():
    plt.annotate(text=row['pop2020'], xy=(row['geometry'].centroid.x, row['geometry'].centroid.y-.12), horizontalalignment='center', fontsize=5)

# Show the plot
plt.show()
